# ArabiKids voices: Chatterbox Multilingual (free GPU)
Makes the Arabic / English / Malay clips with the open-source Chatterbox Multilingual model (MIT licence).

**Before you start:** menu **Runtime > Change runtime type > T4 GPU > Save**.

Then run each cell below in order (press the play button on the left of each one):
1. Install (a few minutes; if Colab says *Restart session*, do it and continue from cell 2)
2. Check the GPU
3. Upload `clips-pilot.json` (later `clips-all.json`)
4. Load the model (first time downloads about 2 GB)
5. Generate
6. Download the zip

Nothing here is approved or live. Judge the Arabic vowel marks (tashkeel) and the Malay by ear first.

In [ ]:
# 1) Install
!pip install -q chatterbox-tts
# Colab's preinstalled torchvision is built for a newer PyTorch than Chatterbox needs and breaks the model import. Chatterbox does not use it.
!pip uninstall -y -q torchvision
# If Colab says "Restart session", do it, then continue from cell 2 (cells 3 and 4 must be run again after a restart).

In [ ]:
# 2) Check the GPU (a GPU is fast; without one the small pilot still works, just slower)
import torch
if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. The 18-clip pilot can still run on the CPU (slow). Do NOT run the full set like this.")
    print("To get a GPU: Runtime > Disconnect and delete runtime, then Change runtime type > T4 GPU > Save > Connect.")

In [ ]:
# 3) Upload the clip list (clips-pilot.json first, clips-all.json after you like the pilot)
import json, os
from google.colab import files
up = files.upload()
name = next(iter(up))
clips = json.loads(up[name].decode("utf-8"))
print(len(clips), "clips from", name)

In [ ]:
# 4) Load the model
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)
try:
    model = ChatterboxMultilingualTTS.from_pretrained(device=device, t3_model="v3")
except TypeError:
    model = ChatterboxMultilingualTTS.from_pretrained(device=device)
print("loaded, sample rate", model.sr)

In [ ]:
# 5) Generate (safe to stop and re-run: finished clips are skipped)
import os, subprocess, torch
import torchaudio as ta

# These come from cells 3 and 4. After a session restart they are gone, so run those again first.
if "clips" not in globals():
    raise SystemExit("Run cell 3 first (upload the clips file).")
if "model" not in globals():
    raise SystemExit("Run cell 4 first (load the model).")

os.makedirs("out", exist_ok=True)

# exaggeration = expressiveness; a lower cfg_weight gives a slower, calmer pace.
# Chatterbox has one built-in voice, so characters differ only by delivery here.
STYLE = {
    "reciter":  dict(exaggeration=0.35, cfg_weight=0.3),
    "narrator": dict(exaggeration=0.5,  cfg_weight=0.4),
    "hudi":     dict(exaggeration=0.7,  cfg_weight=0.4),
    "amir":     dict(exaggeration=0.7,  cfg_weight=0.4),
    "nur":      dict(exaggeration=0.7,  cfg_weight=0.4),
}

def synth(text, lang, style):
    torch.manual_seed(1234)  # same seed for every clip keeps the voice steadier
    try:
        return model.generate(text, language_id=lang, **style)
    except TypeError:
        return model.generate(text, language_id=lang)

done = failed = 0
for i, c in enumerate(clips):
    mp3 = f"out/{c['id']}.mp3"
    if os.path.exists(mp3):
        continue
    try:
        wav = synth(c["text"], c["language"], STYLE.get(c["speaker"], {}))
        ta.save("tmp.wav", wav.cpu(), model.sr)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "tmp.wav", "-ac", "1", "-b:a", "96k", mp3], check=True)
        done += 1
    except Exception as e:
        failed += 1
        print("FAILED", c["id"], repr(e)[:200])
    if (i + 1) % 25 == 0:
        print(f"{i + 1}/{len(clips)}")
print("done", done, "failed", failed)

In [ ]:
# 6) Zip and download (it will not zip an empty folder)
import os, shutil
from google.colab import files
n = len([f for f in os.listdir("out") if f.endswith(".mp3")]) if os.path.isdir("out") else 0
if n == 0:
    raise SystemExit("No clips yet. Run cells 3, 4 and 5 first, and check that cell 5 printed 'done'.")
shutil.make_archive("chatterbox-out", "zip", "out")
print(n, "clips zipped")
files.download("chatterbox-out.zip")